In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import DecisionTreeRegressor, export_graphviz 
from sklearn.model_selection import GridSearchCV,RepeatedStratifiedKFold

In [69]:
census = pd.read_csv('/workspaces/DS-3021/data/CENSUS_ED_ATTN.csv')
census.head()

,A_MARITL,A_SEX,PEAFEVER,PARENT,PENATVTY,PEFNTVTY,PEHSPNON,PEINUSYR,PEPAR1TYP,PRCITSHP,PRDTRACE,ERN_SRCE,WSAL_VAL,ANN_VAL,A_HGA
0,7,1,2,0,57,57,2,0,-1,1,1,1,44200,0,41
1,1,1,2,0,57,57,2,0,-1,1,1,2,0,0,39
2,1,2,2,0,57,57,2,0,-1,1,1,1,48000,0,43
3,1,1,2,0,57,57,2,0,-1,1,1,1,40000,0,40
4,1,2,2,0,57,57,2,0,-1,1,3,1,20000,0,40


In [7]:
census['SOME.COL'] = 0 
# Correct way to create the column based on the condition
census['SOME.COL'] = (census['A_HGA'] > 39).astype(int)


In [5]:
census.dtypes


A_MARITL     int64
A_SEX        int64
PEAFEVER     int64
PARENT       int64
PENATVTY     int64
PEFNTVTY     int64
PEHSPNON     int64
PEINUSYR     int64
PEPAR1TYP    int64
PRCITSHP     int64
PRDTRACE     int64
ERN_SRCE     int64
WSAL_VAL     int64
ANN_VAL      int64
A_HGA        int64
SOME.COL     int64
dtype: object

In [8]:
cols = census.columns.tolist()
cols.remove('WSAL_VAL')
cols.remove('ANN_VAL')
for col in cols:
    census[col] = census[col].astype('category')
census.dtypes

A_MARITL     category
A_SEX        category
PEAFEVER     category
PARENT       category
PENATVTY     category
PEFNTVTY     category
PEHSPNON     category
PEINUSYR     category
PEPAR1TYP    category
PRCITSHP     category
PRDTRACE     category
ERN_SRCE     category
WSAL_VAL        int64
ANN_VAL         int64
A_HGA        category
SOME.COL     category
dtype: object

In [70]:
census.drop(columns=['PRCITSHP', 'PEINUSYR', 'PEPAR1TYP', 'PEAFEVER' ], inplace=True)

In [71]:
census.columns

Index(['A_MARITL', 'A_SEX', 'PARENT', 'PENATVTY', 'PEFNTVTY', 'PEHSPNON',
       'PRDTRACE', 'ERN_SRCE', 'WSAL_VAL', 'ANN_VAL', 'A_HGA'],
      dtype='object')

In [72]:
# Correct way to create the column based on the condition
for row in census.itertuples():
    if row.A_MARITL in [1,2,3,4]:
        census.at[row.Index, 'A_MARITL'] = 1
    elif row.A_MARITL in [5,6]:
        census.at[row.Index, 'A_MARITL'] = 2
    else:
        census.at[row.Index, 'A_MARITL'] = 0

In [86]:
census.PRDTRACE = census.PRDTRACE.apply(lambda x: x if x <= 5 else 6)

In [88]:
X=census.drop(columns=['A_HGA'])
y=census['A_HGA']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42)


In [10]:
kf = RepeatedStratifiedKFold(n_splits=10,n_repeats =5, random_state=42)


In [89]:
scoring = ['precision_macro', 'accuracy']

param={"max_depth":[5,6,7,8,9,0,11,12] }

In [64]:
cl= DecisionTreeClassifier(random_state=1000)

In [79]:
search = GridSearchCV(cl, param, scoring=scoring, n_jobs=-1, cv=kf,refit='precision_macro', verbose=1)

In [90]:
model = search.fit(X_train, y_train)

Fitting 50 folds for each of 6 candidates, totalling 300 fits


/workspaces/DS-3021/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/workspaces/DS-3021/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/workspaces/DS-3021/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/wor

In [91]:
best = search.best_estimator_
print("Best parameters found: ", search.best_params_)

Best parameters found:  {'max_depth': 12}


In [92]:
vals = census['A_HGA'].unique()
unique_values_list = vals.tolist()

print("Unique values in A_HGA: ", unique_values_list)

Unique values in A_HGA:  [41, 39, 43, 40, 44, 42, 0, 33, 34, 37, 45, 46, 35, 36, 32, 38, 31]


In [93]:
varimp=pd.DataFrame(best.feature_importances_,index = X.columns,columns=['importance']).sort_values('importance', ascending=False)
print(varimp)

          importance
PARENT      0.721891
WSAL_VAL    0.118370
ERN_SRCE    0.041625
PENATVTY    0.033226
PEFNTVTY    0.024009
A_SEX       0.017476
A_MARITL    0.017143
PRDTRACE    0.016255
PEHSPNON    0.006356
ANN_VAL     0.003650


In [94]:
best.score(X_test, y_test)

0.4770345963756178

In [95]:
from sklearn.metrics import precision_score

# Predict the labels for the test set
y_pred = best.predict(X_test)

# Calculate the precision_macro score
precision_macro = precision_score(y_test, y_pred, average='macro')

print("Macro Precision Score:", precision_macro)

Macro Precision Score: 0.2567767423753997


In [107]:
census_test = pd.read_csv('/workspaces/DS-3021/data/Census_Test.csv')

In [108]:
# Correct way to create the column based on the condition
for row in census_test.itertuples():
    if row.A_MARITL in [1,2,3,4]:
        census_test.at[row.Index, 'A_MARITL'] = 1
    elif row.A_MARITL in [5,6]:
        census_test.at[row.Index, 'A_MARITL'] = 2
    else:
        census_test.at[row.Index, 'A_MARITL'] = 0

In [109]:
census_test.PRDTRACE = census_test.PRDTRACE.apply(lambda x: x if x <= 5 else 6)

In [110]:
cols = census_test.columns.tolist()
cols.remove('WSAL_VAL')
cols.remove('ANN_VAL')
for col in cols:
    census_test[col] = census_test[col].astype('category')
census_test.dtypes

A_MARITL     category
A_SEX        category
PEAFEVER     category
PARENT       category
PENATVTY     category
PEFNTVTY     category
PEHSPNON     category
PEINUSYR     category
PEPAR1TYP    category
PRCITSHP     category
PRDTRACE     category
ERN_SRCE     category
WSAL_VAL        int64
ANN_VAL         int64
dtype: object

In [111]:
census_test.drop(columns=['PRCITSHP', 'PEINUSYR', 'PEPAR1TYP', 'PEAFEVER' ], inplace=True)

In [112]:
array_output = best.predict(census_test)


In [113]:


# Convert the array to a DataFrame
df = pd.DataFrame(array_output, columns=['A_HGA'])
df['ID']= df.index + 1

# Save the DataFrame to a CSV file
df.to_csv('/workspaces/DS-3021/output.csv', index=False)

print("Array saved to output.csv")

Array saved to output.csv


In [23]:
df

,A_HGA,ID
0,0,0
1,43,1
2,43,2
3,40,3
4,0,4
...,...,...
980,39,980
981,39,981
982,0,982
983,0,983
